# 07c - Affected Term Identification

## Objective

This notebook identifies which **terms in the original text are affected by anonymization**.

This is a preprocessing step for TPI.

The workflow is:

```text
Original text
      +
Anonymized text
      ↓
term-level alignment
      ↓
affected terms in ORIGINAL
      ↓
affected term indices
      ↓
TPI
```

The important distinction is:

> **This notebook identifies what changed.**
> The TPI notebook then quantifies how much information those affected
> original terms contained.

We first implement a deterministic **term-level alignment**. The goal is
to handle the controlled transformations encountered in our anonymization
project:

- replacement;
- deletion;
- insertion;
- multiple modifications;
- phrase deletion;
- punctuation changes.

For the first version, we use a conservative surface definition:

> An original term is considered affected if it cannot be aligned to a
> corresponding preserved term in the anonymized text.

This means that function words such as `in` can be marked as affected when
they disappear because the surrounding structure changes.

We keep this definition explicit rather than silently introducing a semantic
judgment into the alignment procedure.


## 1. Imports and term representation

We use the same `whitespace_terms()` convention as the TPI validation
notebook.

Punctuation is not considered part of a term. For example:

```text
2022. → 2022
```

This is important because affected terms will later be passed directly to
the TPI calculation.


In [14]:
import re
import pandas as pd
from difflib import SequenceMatcher


def whitespace_terms(text):
    """Return terms with character spans, excluding punctuation."""
    return [
        {
            "term": m.group(),
            "start": m.start(),
            "end": m.end(),
        }
        for m in re.finditer(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*", text)
    ]


def term_list(text):
    """Return normalized terms only."""
    return [item["term"] for item in whitespace_terms(text)]


example = "I moved to Geneva in 2022."

display(pd.DataFrame(whitespace_terms(example)))


,term,start,end
0,I,0,1
1,moved,2,7
2,to,8,10
3,Geneva,11,17
4,in,18,20
5,2022,21,25


## 2. Exact term-level alignment

We first align terms using their normalized surface forms.

Example:

```text
Original:     I moved to Geneva in 2022
Anonymized:   I moved to Zurich
```

The common sequence is:

```text
I → I
moved → moved
to → to
```

The remaining original terms are:

```text
Geneva
in
2022
```

and therefore all three are affected under the conservative surface
definition.

`SequenceMatcher` is used only to establish the alignment. It does not
decide whether a replacement is semantically equivalent.


In [15]:
def normalize_term(term):
    """Normalize a term for exact lexical alignment."""
    return term.lower()


def align_terms(original, anonymized):
    """Align original and anonymized terms using lexical sequence alignment."""
    original_terms = whitespace_terms(original)
    anonymized_terms = whitespace_terms(anonymized)

    original_norm = [
        normalize_term(x["term"])
        for x in original_terms
    ]

    anonymized_norm = [
        normalize_term(x["term"])
        for x in anonymized_terms
    ]

    matcher = SequenceMatcher(
        a=original_norm,
        b=anonymized_norm,
        autojunk=False,
    )

    rows = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():

        if tag == "equal":
            for oi, aj in zip(range(i1, i2), range(j1, j2)):
                rows.append({
                    "operation": "equal",
                    "original_index": oi,
                    "original_term": original_terms[oi]["term"],
                    "anonymized_index": aj,
                    "anonymized_term": anonymized_terms[aj]["term"],
                })

        elif tag == "replace":
            original_span = original_terms[i1:i2]
            anonymized_span = anonymized_terms[j1:j2]

            max_len = max(len(original_span), len(anonymized_span))

            for k in range(max_len):
                original_item = (
                    original_span[k]
                    if k < len(original_span)
                    else None
                )
                anonymized_item = (
                    anonymized_span[k]
                    if k < len(anonymized_span)
                    else None
                )

                rows.append({
                    "operation": "replace",
                    "original_index": (
                        i1 + k if original_item is not None else None
                    ),
                    "original_term": (
                        original_item["term"]
                        if original_item is not None
                        else None
                    ),
                    "anonymized_index": (
                        j1 + k if anonymized_item is not None else None
                    ),
                    "anonymized_term": (
                        anonymized_item["term"]
                        if anonymized_item is not None
                        else None
                    ),
                })

        elif tag == "delete":
            for oi in range(i1, i2):
                rows.append({
                    "operation": "delete",
                    "original_index": oi,
                    "original_term": original_terms[oi]["term"],
                    "anonymized_index": None,
                    "anonymized_term": None,
                })

        elif tag == "insert":
            for aj in range(j1, j2):
                rows.append({
                    "operation": "insert",
                    "original_index": None,
                    "original_term": None,
                    "anonymized_index": aj,
                    "anonymized_term": anonymized_terms[aj]["term"],
                })

    return pd.DataFrame(rows)


## 3. Extract affected original terms

For TPI, we are interested specifically in the **original terms** that were
affected.

Under the conservative surface definition:

- `equal` → original term is preserved → **not affected**;
- `replace` → original term is changed → **affected**;
- `delete` → original term disappears → **affected**;
- `insert` → no original term exists → **nothing to add to TPI**.

Therefore, the affected terms are obtained directly from the `replace` and
`delete` operations.


In [16]:
def affected_term_indices_from_alignment(alignment):
    """Return indices of affected terms in the ORIGINAL text."""
    affected = alignment[
        alignment["operation"].isin(["replace", "delete"])
        & alignment["original_index"].notna()
    ]

    return sorted(
        affected["original_index"].astype(int).unique().tolist()
    )


def affected_terms_from_alignment(original, alignment):
    """Return affected original terms in their original order."""
    terms = whitespace_terms(original)
    indices = affected_term_indices_from_alignment(alignment)

    return [
        terms[i]["term"]
        for i in indices
    ]


def identify_affected_terms(original, anonymized):
    """Complete alignment + affected-term identification pipeline."""
    alignment = align_terms(original, anonymized)

    affected_indices = affected_term_indices_from_alignment(alignment)
    affected_terms = affected_terms_from_alignment(
        original,
        alignment,
    )

    return {
        "alignment": alignment,
        "affected_indices": affected_indices,
        "affected_terms": affected_terms,
    }


## 4. Basic examples

### Example 1 — replacement

```text
I live in Lausanne.
I live in Geneva.
```

`Lausanne` is affected.

### Example 2 — deletion

```text
I moved to Geneva in 2022.
I moved to Geneva.
```

`2022` is affected.

### Example 3 — replacement + deletion

```text
I moved to Geneva in 2022.
I moved to Zurich.
```

Under the surface definition, the affected original terms are:

```text
Geneva, in, 2022
```

The word `in` is included because it disappears from the anonymized text.


In [17]:
examples = [
    {
        "name": "Replacement",
        "original": "I live in Lausanne.",
        "anonymized": "I live in Geneva.",
    },
    {
        "name": "Deletion",
        "original": "I moved to Geneva in 2022.",
        "anonymized": "I moved to Geneva.",
    },
    {
        "name": "Replacement + deletion",
        "original": "I moved to Geneva in 2022.",
        "anonymized": "I moved to Zurich.",
    },
]

for example in examples:
    result = identify_affected_terms(
        example["original"],
        example["anonymized"],
    )

    print("\n" + example["name"])
    print("Original:  ", example["original"])
    print("Anonymized:", example["anonymized"])
    print("Affected:  ", result["affected_terms"])
    display(result["alignment"])



Replacement
Original:   I live in Lausanne.
Anonymized: I live in Geneva.
Affected:   ['Lausanne']


,operation,original_index,original_term,anonymized_index,anonymized_term
0,equal,0,I,0,I
1,equal,1,live,1,live
2,equal,2,in,2,in
3,replace,3,Lausanne,3,Geneva



Deletion
Original:   I moved to Geneva in 2022.
Anonymized: I moved to Geneva.
Affected:   ['in', '2022']


,operation,original_index,original_term,anonymized_index,anonymized_term
0,equal,0,I,0.0,I
1,equal,1,moved,1.0,moved
2,equal,2,to,2.0,to
3,equal,3,Geneva,3.0,Geneva
4,delete,4,in,NaN,NaN
5,delete,5,2022,NaN,NaN



Replacement + deletion
Original:   I moved to Geneva in 2022.
Anonymized: I moved to Zurich.
Affected:   ['Geneva', 'in', '2022']


,operation,original_index,original_term,anonymized_index,anonymized_term
0,equal,0,I,0.0,I
1,equal,1,moved,1.0,moved
2,equal,2,to,2.0,to
3,replace,3,Geneva,3.0,Zurich
4,replace,4,in,NaN,NaN
5,replace,5,2022,NaN,NaN


## 5. Visualize the affected original terms

The following helper highlights the original terms identified as affected.

This is useful for debugging the alignment before sending the indices to TPI.


In [18]:
def highlight_affected_terms(text, affected_indices):
    """Return a readable representation of the original text with affected terms marked."""
    terms = whitespace_terms(text)
    affected_indices = set(affected_indices)

    pieces = []
    cursor = 0

    for i, item in enumerate(terms):
        pieces.append(text[cursor:item["start"]])

        if i in affected_indices:
            pieces.append(f"[[{item['term']}]]")
        else:
            pieces.append(item["term"])

        cursor = item["end"]

    pieces.append(text[cursor:])

    return "".join(pieces)


for example in examples:
    result = identify_affected_terms(
        example["original"],
        example["anonymized"],
    )

    print(example["name"])
    print(
        highlight_affected_terms(
            example["original"],
            result["affected_indices"],
        )
    )
    print()


Replacement
I live in [[Lausanne]].

Deletion
I moved to Geneva [[in]] [[2022]].

Replacement + deletion
I moved to [[Geneva]] [[in]] [[2022]].



## 6. Controlled benchmark

We now test the alignment on the main transformation types expected in the
project.

The benchmark is deliberately simple: the expected affected terms are known
beforehand, so we can test whether the alignment recovers them.


In [19]:
benchmark = [
    {
        "name": "No change",
        "original": "I live in Geneva.",
        "anonymized": "I live in Geneva.",
        "expected": [],
    },
    {
        "name": "Single replacement",
        "original": "I live in Lausanne.",
        "anonymized": "I live in Geneva.",
        "expected": ["Lausanne"],
    },
    {
        "name": "Single deletion",
        "original": "I live in Geneva since 2022.",
        "anonymized": "I live in Geneva.",
        "expected": ["since", "2022"],
    },
    {
        "name": "Replacement + deletion",
        "original": "I moved to Geneva in 2022.",
        "anonymized": "I moved to Zurich.",
        "expected": ["Geneva", "in", "2022"],
    },
    {
        "name": "Phrase replacement",
        "original": "I have two brothers.",
        "anonymized": "I have siblings.",
        "expected": ["two", "brothers"],
    },
    {
    "name": "Multiple modifications",
    "original": "I moved from Brussels to Geneva in 2022.",
    "anonymized": "I moved from Zurich to Bern.",
    "expected": ["Brussels", "Geneva", "in", "2022"],
    },
]

benchmark_rows = []

for case in benchmark:
    result = identify_affected_terms(
        case["original"],
        case["anonymized"],
    )

    detected = result["affected_terms"]
    expected = case["expected"]

    benchmark_rows.append({
        "case": case["name"],
        "original": case["original"],
        "anonymized": case["anonymized"],
        "expected": expected,
        "detected": detected,
        "match": detected == expected,
    })

benchmark_df = pd.DataFrame(benchmark_rows)
display(benchmark_df)

assert benchmark_df["match"].all(), (
    "At least one controlled alignment test did not match the expected "
    "affected terms."
)

print("✓ Controlled benchmark passed.")


,case,original,anonymized,expected,detected,match
0,No change,I live in Geneva.,I live in Geneva.,[],[],True
1,Single replacement,I live in Lausanne.,I live in Geneva.,[Lausanne],[Lausanne],True
2,Single deletion,I live in Geneva since 2022.,I live in Geneva.,"[since, 2022]","[since, 2022]",True
3,Replacement + deletion,I moved to Geneva in 2022.,I moved to Zurich.,"[Geneva, in, 2022]","[Geneva, in, 2022]",True
4,Phrase replacement,I have two brothers.,I have siblings.,"[two, brothers]","[two, brothers]",True
5,Multiple modifications,I moved from Brussels to Geneva in 2022.,I moved from Zurich to Bern.,"[Brussels, Geneva, in, 2022]","[Brussels, Geneva, in, 2022]",True


✓ Controlled benchmark passed.


## 7. Important limitation: lexical alignment

The current method is intentionally simple.

`SequenceMatcher` is a **surface lexical alignment**. It does not understand
meaning.

For example:

```text
Original:
I live in Lausanne.

Anonymized:
I reside in Lausanne.
```

The method can identify that `live` and `reside` differ, but it does not know
that they have similar meanings.

Likewise, in:

```text
Original:
I moved to Geneva in 2022.

Anonymized:
I moved to Zurich.
```

the alignment can identify the changed surface material, but it cannot decide
whether `in` should be treated as semantically affected or merely as a
grammatical consequence of the transformation.

This notebook therefore defines the current output as:

> **surface-affected original terms**

A later semantic alignment variant can be evaluated separately if needed.


## 8. From affected terms to TPI

Once the affected original indices are identified, they can be passed directly
to the TPI implementation from `07a`.

Conceptually:

```text
Original + anonymized
        ↓
07b: term alignment
        ↓
affected original indices
        ↓
07a: contextual IC estimation
        ↓
information lost
        ↓
TPI
```

For example:

```python
result = identify_affected_terms(
    "I moved to Geneva in 2022.",
    "I moved to Zurich.",
)

result["affected_indices"]
# → [3, 4, 5]

result["affected_terms"]
# → ["Geneva", "in", "2022"]
```

These indices are exactly what `tpi_from_spans()` expects.


In [20]:
# Integration example.
# This cell assumes the TPI functions from 07a have been run in the same
# environment/notebook session.

original = "I moved to Geneva in 2022."
anonymized = "I moved to Zurich."

alignment_result = identify_affected_terms(
    original,
    anonymized,
)

print("Affected indices:", alignment_result["affected_indices"])
print("Affected terms:", alignment_result["affected_terms"])

print("\nIf 07a is loaded in the same environment, TPI can be computed with:")
print(
    "tpi_from_spans(ic_table, alignment_result['affected_indices'])"
)


Affected indices: [3, 4, 5]
Affected terms: ['Geneva', 'in', '2022']

If 07a is loaded in the same environment, TPI can be computed with:
tpi_from_spans(ic_table, alignment_result['affected_indices'])


## 9. Next step

This notebook validates the **surface term alignment** required before TPI.

The next step is to test the alignment on a larger set of realistic
anonymization examples and determine whether the conservative surface
definition is sufficient.

In particular, we should investigate:

1. multi-word entities;
2. reordered words;
3. paraphrases;
4. insertions and deletions inside a phrase;
5. punctuation and contractions;
6. cases where anonymization changes grammatical structure;
7. whether a semantic alignment method is needed in addition to lexical
   alignment.

Only after this validation should the alignment procedure be applied to the
real interview corpus.


## 10. Robustness tests

The controlled benchmark validates basic replacement and deletion cases.
We now test situations that are more challenging for lexical alignment:

1. insertion of new words;
2. reordering of words;
3. multi-word insertion;
4. phrase deletion;
5. one-to-many and many-to-one changes;
6. punctuation changes.

A mismatch is not automatically a failure of the whole approach. It is a
signal that the current lexical alignment needs to be examined before being
used on real interview data.


In [21]:
robustness_cases = [
    {
        "name": "Insertion",
        "original": "I live in Geneva.",
        "anonymized": "I currently live in Geneva.",
        "expected": [],
    },
    {
        "name": "Reordering",
        "original": "I live in Geneva.",
        "anonymized": "In Geneva I live.",
        "expected": [],
    },
    {
        "name": "Multi-word insertion",
        "original": "I live in Geneva.",
        "anonymized": "I live in the city of Geneva.",
        "expected": [],
    },
    {
        "name": "Phrase deletion",
        "original": "I live in Geneva with my brother.",
        "anonymized": "I live in Geneva.",
        "expected": ["with", "my", "brother"],
    },
    {
        "name": "One-to-many replacement",
        "original": "I live in Geneva.",
        "anonymized": "I live in the Swiss city of Geneva.",
        "expected": [],
    },
    {
        "name": "Many-to-one replacement",
        "original": "I live in the Swiss city.",
        "anonymized": "I live in Geneva.",
        "expected": ["the", "Swiss", "city"],
    },
    {
        "name": "Punctuation only",
        "original": "I live in Geneva.",
        "anonymized": "I live in Geneva!",
        "expected": [],
    },
]

robustness_rows = []

for case in robustness_cases:
    result = identify_affected_terms(
        case["original"],
        case["anonymized"],
    )

    robustness_rows.append({
        "case": case["name"],
        "original": case["original"],
        "anonymized": case["anonymized"],
        "expected": case["expected"],
        "detected": result["affected_terms"],
        "match": result["affected_terms"] == case["expected"],
    })

robustness_df = pd.DataFrame(robustness_rows)
display(robustness_df)


,case,original,anonymized,expected,detected,match
0,Insertion,I live in Geneva.,I currently live in Geneva.,[],[],True
1,Reordering,I live in Geneva.,In Geneva I live.,[],"[in, Geneva]",False
2,Multi-word insertion,I live in Geneva.,I live in the city of Geneva.,[],[],True
3,Phrase deletion,I live in Geneva with my brother.,I live in Geneva.,"[with, my, brother]","[with, my, brother]",True
4,One-to-many replacement,I live in Geneva.,I live in the Swiss city of Geneva.,[],[],True
5,Many-to-one replacement,I live in the Swiss city.,I live in Geneva.,"[the, Swiss, city]","[the, Swiss, city]",True
6,Punctuation only,I live in Geneva.,I live in Geneva!,[],[],True


## 11. Inspecting robustness cases

We deliberately do not force all robustness cases to pass. A mismatch tells
us where lexical sequence alignment may be insufficient.


In [22]:
for _, row in robustness_df.iterrows():
    if not row["match"]:
        print("=" * 80)
        print("CASE:", row["case"])
        print("Original:  ", row["original"])
        print("Anonymized:", row["anonymized"])
        print("Expected:  ", row["expected"])
        print("Detected:  ", row["detected"])

        result = identify_affected_terms(
            row["original"],
            row["anonymized"],
        )

        display(result["alignment"])

        print(
            "Highlighted original:",
            highlight_affected_terms(
                row["original"],
                result["affected_indices"],
            ),
        )


CASE: Reordering
Original:   I live in Geneva.
Anonymized: In Geneva I live.
Expected:   []
Detected:   ['in', 'Geneva']


,operation,original_index,original_term,anonymized_index,anonymized_term
0,insert,NaN,NaN,0.0,In
1,insert,NaN,NaN,1.0,Geneva
2,equal,0.0,I,2.0,I
3,equal,1.0,live,3.0,live
4,delete,2.0,in,NaN,NaN
5,delete,3.0,Geneva,NaN,NaN


Highlighted original: I live [[in]] [[Geneva]].


## 12. What the robustness tests tell us

### Insertions

If new words are added but every original term is preserved, those inserted
words should not contribute to TPI because TPI evaluates information from the
original text.

### Reordering

If all original terms are preserved but their order changes, a sequence-based
alignment may incorrectly mark terms as affected. This is a key limitation to
test.

### Multi-word changes

A replacement may involve one original term and several anonymized terms, or
several original terms and one anonymized term. This is why the final output
should be interpreted as **affected original spans**, not simply one-to-one
word pairs.

### Punctuation

Punctuation changes should not create affected terms because punctuation is
excluded from the term representation.


In [23]:
# Checks that should hold independently of the ambiguous cases.

punctuation_result = identify_affected_terms(
    "I live in Geneva.",
    "I live in Geneva!",
)
assert punctuation_result["affected_terms"] == []
print("✓ Punctuation-only change does not affect original terms.")

insertion_result = identify_affected_terms(
    "I live in Geneva.",
    "I currently live in Geneva.",
)
assert insertion_result["affected_terms"] == []
print("✓ Pure insertion does not affect original terms.")


✓ Punctuation-only change does not affect original terms.
✓ Pure insertion does not affect original terms.


## 13. Decision point

The basic benchmark validates the implementation for straightforward
replacement and deletion.

The robustness tests determine whether `SequenceMatcher` is sufficiently
reliable for the transformations expected in the interview anonymization
data.

The cases requiring particular attention are:

- word reordering;
- one-to-many replacements;
- many-to-one replacements;
- multi-word entities;
- grammatical restructuring.

If these cases systematically create false affected terms, we should replace
the current sequence-level alignment with a more explicit span-alignment
method before applying TPI to the real corpus.

The required output remains:

```python
affected_indices
```

where the indices always refer to terms in the **original** text.


## 14. Conclusion

We now validate affected-term identification at two levels:

1. **Controlled cases** — straightforward replacements, deletions, and
   multiple modifications.
2. **Robustness cases** — insertions, reordering, multi-word changes,
   phrase deletion, and punctuation.

The objective is not to make every robustness case pass by construction.
Instead, we use these cases to identify the situations in which lexical
alignment is reliable and the situations where a more sophisticated
alignment method is required.
